# Experiment report

Generated by `tools/reports` (§14.1). Every panel reads the per-run SQLite DB (§13) through `tools/reports/panels.py`.

In [ ]:
# parameters (overridden by tools/reports/run.py)
DB_PATH = "run.db"
OUT_DIR = "."
# §14.1 supportable-users estimate — editable per-class user-behaviour model.
# PROVISIONAL defaults; see TODOs.md "Per-class sessions_per_user_per_hour defaults".
SESSIONS_PER_USER_PER_HOUR = {
    "agentic-coding": 4.0,        # agentic tasks per developer per active hour
    "chat-short-turns": 2.0,      # chat conversations per user per active hour
    "long-context-followup": 1.0,
    "smoke-synthetic": 1.0,
}

In [ ]:
import json
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from tools.reports import panels

pd.set_option("display.width", 160)
conn = panels.connect(DB_PATH)
exp = panels.experiment(conn)
out_dir = Path(OUT_DIR)
print(f"run: {exp['run_id']}  model: {exp['model']}  backend: {exp['backend']}")

## Scenario & assumptions (§13.7)

Read every chart below in the context of what each workload class models — and, **prominently**, what it does *not*.

In [ ]:
manifest = exp.get("scenario_manifest") or {}
print("MIX:")
for m in manifest.get("mix", exp.get("scenario_mix") or []):
    print(f"  {m['scenario'] if 'scenario' in m else m.get('name')}: weight={m['weight']}"
          + (f", expected request share={m['expected_request_share']:.0%}" if "expected_request_share" in m else ""))
for cls in manifest.get("classes", []):
    print(f"\n[{cls['name']}] ({cls['maturity']}) — {cls['summary']}")
    for item in cls.get("modelled", []):
        print(f"  + modelled: {item}")
    for item in cls.get("not_modelled", []):
        print(f"  ! NOT MODELLED: {item}")          # §14.1: visually distinguished
    for item in cls.get("assumptions", []):
        print(f"  ~ assumes: {item}")
print("\nRUN ASSUMPTIONS:")
for item in manifest.get("run_assumptions", []):
    print(f"  ~ {item}")

## System pre-checks (§13.6) — warns/fails flagged

In [ ]:
prechecks = panels.prechecks_table(conn)
if prechecks.empty:
    print("no pre-check rows recorded")
else:
    flagged = prechecks[prechecks.status != "pass"]
    if not flagged.empty:
        print("!!! DEGRADED FOUNDATION — interpret all numbers below with care !!!")
    display(prechecks)

## Model loading times (§9.2)

In [ ]:
display(panels.model_load_table(conn))

## Latency vs λ (§14.1) — per-class SLO lines from §12.4

In [ ]:
def latency_plot(metric, fname):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    overall = panels.latency_percentiles(conn, metric)
    for pct in ("p50", "p95", "p99"):
        ax.plot(overall.rate_lambda, overall[pct], marker="o", label=pct)
    for slo in exp.get("slos") or []:
        if slo["metric"] == metric:
            ax.axhline(slo["threshold"], linestyle="--", color="red", alpha=0.6,
                       label=f"SLO {slo['scenario']} {slo.get('percentile','')} ≤ {slo['threshold']}")
    ax.set_xscale("log"); ax.set_xlabel("λ (session starts/s, §11.3)"); ax.set_ylabel(f"{metric}")
    ax.legend(); ax.set_title(f"{metric} vs λ")
    fig.savefig(out_dir / fname, dpi=120, bbox_inches="tight")
    plt.show()

latency_plot("ttft_ms", "ttft.png")
latency_plot("tpot_ms", "itl.png")

## Per-class breakdown (§12.2/§14.1) — interference made visible

In [ ]:
per_class = panels.latency_percentiles(conn, "ttft_ms", per_class=True)
display(per_class)
fig, ax = plt.subplots(figsize=(8, 4.5))
for cls, group in per_class.groupby("scenario"):
    ax.plot(group.rate_lambda, group.p95, marker="o", label=f"{cls} p95")
ax.set_xscale("log"); ax.set_xlabel("λ (session starts/s)"); ax.set_ylabel("ttft_ms p95")
ax.legend(); ax.set_title("per-class TTFT p95 vs λ")
fig.savefig(out_dir / "ttft_per_class.png", dpi=120, bbox_inches="tight")
plt.show()
display(panels.error_rates(conn, per_class=True))

## SLO attainment and λ\* (§12.4)

In [ ]:
slo_table, lambda_star = panels.slo_attainment(conn)
if slo_table.empty:
    print("no SLOs declared")
else:
    display(slo_table)
print(f"λ* = {lambda_star}" if lambda_star is not None
      else "λ* UNDEFINED — no swept λ meets all objectives (§12.4); extend the sweep toward lower rates")

## Supportable-users estimate (§14.1)

Computed **only here** — an estimate, parameters disclosed above.

In [ ]:
est = panels.users_estimate(conn, SESSIONS_PER_USER_PER_HOUR)
if est is None:
    print("undefined: λ* is undefined")
else:
    display(est)
    for _, row in est.iterrows():
        if row.supportable_user_population:
            print(f"{row.scenario}: ≈{row.supportable_user_population:,.0f} users "
                  f"(at {row.sessions_per_user_per_hour} sessions/user/h), "
                  f"{row.concurrent_active_sessions:.1f} concurrent active sessions")

## Response quality (§12.5) — gate outcome + Stage-B scores

In [ ]:
quality = panels.quality_table(conn)
if quality.empty:
    print("no quality evaluations recorded")
else:
    gate = quality[quality.stage == "gate"]
    if (gate.status == "fail").any():
        print("!!! QUALITY-FLAGGED: the sanity gate FAILED (§12.5) !!!")
    display(quality)
    # cross-config capacity-vs-quality pairing spans runs — assembled in
    # curated reports (§14.3) from the centralized DB; this panel shows this run.

## Hardware headroom (§12.3/§14.1)

In [ ]:
hw = panels.hardware_summary(conn)
if hw.empty:
    print("no hardware telemetry recorded")
else:
    display(hw)

## Raw per-rate-level table

In [ ]:
display(panels.raw_rate_table(conn))
conn.close()